# Phase 8: Digital Twin and Synthetic Telemetry Simulation
## AI-Driven Intelligent Greywater Management and Smart Reuse Routing System

### Section 1: Digital Twin Objective
The objective of Phase 8 is to construct a virtual Digital Twin representing a physical greywater recycling facility.
The Digital Twin continuously maintains state for:
- Source generation dynamics (Bathroom, Kitchen, Laundry, Mixed)
- 15 physicochemical and biological water-quality sensors
- Real-time inflow and physical storage tank level
- Measurement noise and hardware sensor faults
- Controlled synthetic contamination shock events
- Downstream inference adapter connecting directly to the trained Isolation Forest and Optimized XGBoost models.

> **Important Operational Disclaimer:**  
> The Digital Twin telemetry is synthetic simulation data derived from the statistical characteristics of the supplied dataset. It is not a substitute for physical sensor measurements.


### Section 2: Reference Dataset & Empirical Profiles
Inspect the source-specific statistical profiles computed from the project training dataset.


In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

with open('../simulation/source_statistical_profiles.json', 'r') as f:
    profiles = json.load(f)

print("Source Statistical Profile Categories:", list(profiles.keys()))
print("\nBathroom Empirical Profile Summary (Sample):")
print(pd.DataFrame(profiles['Bathroom']).T[['mean', 'std', 'min', 'max']].head(6))


### Section 3: Simulation Configuration
Review the centralized simulation parameters defined in `simulation/config.py`.


In [ ]:
from simulation.config import (
    DEFAULT_RANDOM_SEED,
    DEFAULT_SIMULATION_INTERVAL_MIN,
    DEFAULT_TANK_CAPACITY_L,
    DEFAULT_INITIAL_TANK_LEVEL_L,
    FLOW_RATE_RANGES_L_MIN,
    SYNTHETIC_EVENTS
)

print(f"Random Seed: {DEFAULT_RANDOM_SEED}")
print(f"Interval: {DEFAULT_SIMULATION_INTERVAL_MIN} minutes")
print(f"Tank Capacity: {DEFAULT_TANK_CAPACITY_L} L | Initial Level: {DEFAULT_INITIAL_TANK_LEVEL_L} L")
print(f"Available Synthetic Event Scenarios: {list(SYNTHETIC_EVENTS.keys())}")


### Section 4: Initial System State
Instantiate the `DigitalTwinEngine` and inspect the initial state at step 0.


In [ ]:
from simulation.digital_twin import DigitalTwinEngine, DigitalTwinState

engine = DigitalTwinEngine(random_seed=42)
initial_state = engine.create_initial_state(start_timestamp="2026-10-02 06:00:00", initial_source="Bathroom")

print("Initial Digital Twin State:")
for k, v in initial_state.to_dict().items():
    print(f"  {k}: {v}")


### Section 5: Telemetry Generation Mechanism
Demonstrate base telemetry generation from empirical source distributions.


In [ ]:
kitchen_telemetry = engine._generate_base_telemetry("Kitchen")
bathroom_telemetry = engine._generate_base_telemetry("Bathroom")

comp_df = pd.DataFrame({
    "Bathroom (Cleanest)": bathroom_telemetry,
    "Kitchen (Contaminated)": kitchen_telemetry
})
print("Source Parameter Contrast (Kitchen vs Bathroom):")
print(comp_df.loc[['pH', 'TUR_NTU', 'COD_mg_L', 'BOD_mg_L', 'E_coli_CFU_100mL']])


### Section 6: Temporal Simulation & State Progression
Advance the Digital Twin step-by-step across consecutive timesteps.


In [ ]:
s0 = engine.current_state
s1 = engine.update_state(source="Bathroom", event_name="normal")
s2 = engine.update_state(source="Kitchen", event_name="normal")

print(f"Step 0: {s0.timestamp} | Source: {s0.greywater_source} | Tank: {s0.tank_level_L:.1f} L")
print(f"Step 1: {s1.timestamp} | Source: {s1.greywater_source} | Tank: {s1.tank_level_L:.1f} L")
print(f"Step 2: {s2.timestamp} | Source: {s2.greywater_source} | Tank: {s2.tank_level_L:.1f} L")


### Section 7: Greywater Inflow Dynamics
Verify that flow rates are non-negative and remain within source-specific operational bounds.


In [ ]:
flows = [engine._generate_flow_rate("Laundry") for _ in range(5)]
print("Sampled Laundry Inflow Rates (L/min):", flows)
assert all(f >= 0.0 for f in flows), "Flow rate cannot be negative!"
print("Flow rate constraint validated: Non-negative fluid generation.")


### Section 8: Physical Storage Tank Mass Balance
Verify that the storage tank level accounts for inflows and outflows and clamps strictly within $[0, \text{Capacity}]$. 


In [ ]:
level_mid = engine.update_tank(current_level=500.0, inflow_rate_L_min=20.0, outflow_rate_L_min=10.0)
print(f"Nominal Net Inflow: 500 L + (20 - 10) * 5 min = {level_mid} L")

level_overflow = engine.update_tank(current_level=980.0, inflow_rate_L_min=100.0, outflow_rate_L_min=0.0)
print(f"Overflow Clamping: 980 L + 500 L = {level_overflow} L (Capped at {engine.tank_capacity_L} L)")

level_empty = engine.update_tank(current_level=30.0, inflow_rate_L_min=0.0, outflow_rate_L_min=50.0)
print(f"Empty Clamping: 30 L - 250 L = {level_empty} L (Floored at 0.0 L)")


### Section 9: Measurement Sensor Noise
Demonstrate bounded sensor noise addition to simulate real-world transducer jitter.


In [ ]:
base = {"pH": 7.0, "TUR_NTU": 35.0, "COD_mg_L": 120.0}
noisy = engine.apply_sensor_noise(base)
print("Base Reading:", base)
print("Noisy Reading:", noisy)


### Section 10: Sensor Hardware Fault Simulation
Inject stuck sensor and calibration drift hardware faults.


In [ ]:
stuck_telemetry, status_stuck = engine.apply_sensor_fault({"TUR_NTU": 35.0}, "sensor_fault_stuck")
print(f"Stuck Sensor Simulation: Value={stuck_telemetry['TUR_NTU']}, Status='{status_stuck}'")

drift_telemetry, status_drift = engine.apply_sensor_fault({"pH": 7.0}, "sensor_fault_drift")
print(f"Sensor Drift Simulation: Value={drift_telemetry['pH']:.2f}, Status='{status_drift}'")


### Section 11: Controlled Synthetic Contamination Events
Inject synthetic testing shocks (turbidity shock, high organic load, microbial spike, compound event).


In [ ]:
turbid_shock = engine.apply_event({"TUR_NTU": 35.0, "TSS_mg_L": 50.0}, "high_turbidity")
print("Injected High Turbidity Shock:", turbid_shock)

organic_shock = engine.apply_event({"COD_mg_L": 100.0, "BOD_mg_L": 40.0}, "high_organic_load")
print("Injected High Organic Shock:", organic_shock)


### Section 12: Telemetry Time-Series Visualization
Inspect the 24-hour simulation data generated by `simulation/run_simulation.py`.


In [ ]:
sim_df = pd.read_csv('../simulation/data/synthetic_telemetry.csv')
print(f"Loaded {len(sim_df)} synthetic telemetry timesteps.")

plt.figure(figsize=(12, 4))
plt.plot(pd.to_datetime(sim_df['timestamp']), sim_df['tank_level_L'], color='#2563eb', linewidth=2, label='Tank Level (L)')
plt.axhline(1000.0, color='r', linestyle='--', label='Max Capacity (1000 L)')
plt.ylabel('Volume (L)')
plt.title('Digital Twin 24-Hour Tank Volume Dynamics')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


### Section 13: Model Inference Integration (No Retraining)
Send simulated telemetry through the `ModelInferenceAdapter` to evaluate Isolation Forest and XGBoost in real time.


In [ ]:
from simulation.model_inference import ModelInferenceAdapter

adapter = ModelInferenceAdapter()
sample_state = engine.create_initial_state(initial_source="Kitchen")

inf_result = adapter.predict(sample_state)
print("Real-Time Telemetry Inference Result:")
for k, v in inf_result.items():
    print(f"  {k}: {v}")


### Section 14: Automated Data Quality Validation
Review the data quality checklist created in `reports/digital_twin_validation.md`.


In [ ]:
with open('../reports/digital_twin_validation.md', 'r') as f:
    report_text = f.read()
print(report_text)


### Section 15: Conclusions & Operational Limitations
1. **Simulation Fidelity:** The Digital Twin successfully mirrors source-specific empirical distributions, non-negative fluid dynamics, and tank mass balances.
2. **Model Decoupling:** Telemetry connects to the saved models purely through inference without modifying serialized weights or retraining.
3. **Safety Arbitration:** Anomaly screening and physical hazard rules intercept simulated contamination shocks (turbidity, organic, pathogen) and apply safety overrides.
4. **Mandatory Scientific Limitation:**
   > *"The Digital Twin telemetry is synthetic simulation data derived from the statistical characteristics of the supplied dataset. It is not a substitute for physical sensor measurements."*
